# Elaborative Rehearsal (B) — Data Prep (stage 1)

- **input** = source article truncated to `max_words` (350).
- **target** = XSum one-sentence summary (abstractive).

Mirrors `04`: A's target is verbatim source sentences, B's target mostly is not.

XSum (not cnn_dailymail): cnn_dailymail highlights are near-extractive, would teach copying. XSum forces abstraction — makes `07`'s manipulation check (novel n-gram ratio high for B, ~0 for A) meaningful.

Stage 1 (this + `07`) teaches abstraction on single documents only, no cross-chunk integration. Stage 2 (rolling curation, not yet built here) does that.

| Choice | Source |
|---|---|
| Small seq2seq compressor, distilled from teacher | RECOMP (Xu, Shi & Choi, ICLR 2024) §3.2 |
| Gist over truncation | ReadAgent (Lee et al., 2024) |
| `(prev summary, new context) → updated summary` | arXiv:2308.15022 |
| "Revise, don't append" | C-DIC (ICML 2026) |


In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import pandas as pd
import yaml
from datasets import load_dataset
from transformers import AutoTokenizer

from src.pipeline.rehearsal import novel_ngram_ratio

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load `EdinburghNLP/xsum`

Official train/validation/test split, same discipline as `04`.


In [2]:
MAX_TRAIN_EXAMPLES = 3000
MAX_VAL_EXAMPLES = 300
MAX_TEST_EXAMPLES = 300

train_raw = load_dataset("EdinburghNLP/xsum", split=f"train[:{MAX_TRAIN_EXAMPLES}]")
val_raw = load_dataset("EdinburghNLP/xsum", split=f"validation[:{MAX_VAL_EXAMPLES}]")
test_raw = load_dataset("EdinburghNLP/xsum", split=f"test[:{MAX_TEST_EXAMPLES}]")
print(f"train: {len(train_raw)}, validation: {len(val_raw)}, test: {len(test_raw)}")

sample = train_raw[0]
print("\nsample keys:", list(sample.keys()))
print("document (first 300 chars):", sample["document"][:300])
print("\nsummary:", sample["summary"])

train: 3000, validation: 300, test: 300

sample keys: ['document', 'summary', 'id']
document (first 300 chars): The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.
Trains on the west coast mainline face disruption due to damage at the Lamington Viaduct.
Many

summary: Clean-up operations are continuing across the Scottish Borders and Dumfries and Galloway after flooding caused by Storm Frank.


## 2. Build (input, target) pairs

No oracle step — XSum's target is already abstractive.


In [3]:
CHUNK_MAX_WORDS = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]


def truncate_words(text: str, max_words: int) -> str:
    return " ".join(text.split()[:max_words])


def build_pair(example: dict) -> dict | None:
    truncated = truncate_words(example["document"], CHUNK_MAX_WORDS)
    summary = example["summary"].strip()
    if not truncated.strip() or not summary:
        return None
    return {"id": example["id"], "input_text": truncated, "target_text": summary}


train_pairs = [p for p in (build_pair(ex) for ex in train_raw) if p is not None]
val_pairs = [p for p in (build_pair(ex) for ex in val_raw) if p is not None]
test_pairs = [p for p in (build_pair(ex) for ex in test_raw) if p is not None]
print(f"train pairs: {len(train_pairs)} / {len(train_raw)}")
print(f"val pairs:   {len(val_pairs)} / {len(val_raw)}")
print(f"test pairs:  {len(test_pairs)} / {len(test_raw)}")

pd.DataFrame(train_pairs)[["input_text", "target_text"]].head(3)

train pairs: 3000 / 3000
val pairs:   299 / 300
test pairs:  300 / 300


,input_text,target_text
0,"The full cost of damage in Newton Stewart, one...",Clean-up operations are continuing across the ...
1,A fire alarm went off at the Holiday Inn in Ho...,Two tourist buses have been destroyed by fire ...
2,Ferrari appeared in a position to challenge un...,Lewis Hamilton stormed to pole position at the...


### Truncation sanity check + the abstractiveness that motivates B

Measures content survival and novelty of the summary vs the truncated input.


In [4]:
import re

_STOP = set(
    "a an the and or but if of to in on at for with by from as is are was were be been it its this that "
    "he she they them his her their has have had will would can could not no".split()
)


def content_overlap(summary: str, document: str) -> float:
    """Fraction of the summary's content words that appear in the document."""
    words = [w for w in re.findall(r"[a-z']+", summary.lower()) if w not in _STOP]
    if not words:
        return 1.0
    doc = set(re.findall(r"[a-z']+", document.lower()))
    return sum(w in doc for w in words) / len(words)


sample_pairs = train_pairs[:300]
overlaps = [content_overlap(p["target_text"], p["input_text"]) for p in sample_pairs]
novelty = [novel_ngram_ratio(p["target_text"], p["input_text"], n=3) for p in sample_pairs]

print(f"on {len(sample_pairs)} training pairs (after 350-word truncation):")
print(f"  summary content words present in input : mean {sum(overlaps) / len(overlaps):.3f}")
print(f"  target novel 3-gram ratio              : mean {sum(novelty) / len(novelty):.3f}")
print("\n(A's oracle targets are extracted from the input, so their novel 3-gram")
print(" ratio is 0.0 by construction. B's targets should be far above that —")
print(" that gap is the manipulation the experiment claims to make.)")

on 300 training pairs (after 350-word truncation):
  summary content words present in input : mean 0.427
  target novel 3-gram ratio              : mean 0.980

(A's oracle targets are extracted from the input, so their novel 3-gram
 ratio is 0.0 by construction. B's targets should be far above that —
 that gap is the manipulation the experiment claims to make.)


## 3. Tokenize

`INPUT_MAX_LENGTH` matches `04`; target length is much shorter (one-sentence summaries).


In [5]:
INPUT_MAX_LENGTH = 512
TARGET_MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained("t5-small")

for name, pairs in [("input", "input_text"), ("target", "target_text")]:
    lengths = sorted(len(tokenizer(p[pairs]).input_ids) for p in train_pairs)
    pct = lambda q: lengths[int(q * (len(lengths) - 1))]  # noqa: E731
    print(f"{name:7} token length  50th={pct(0.50):>4}  90th={pct(0.90):>4}  99th={pct(0.99):>4}")


def tokenize_pairs(pairs: list[dict]) -> datasets.Dataset:
    inputs = tokenizer([p["input_text"] for p in pairs], max_length=INPUT_MAX_LENGTH, truncation=True)
    targets = tokenizer([p["target_text"] for p in pairs], max_length=TARGET_MAX_LENGTH, truncation=True)
    return datasets.Dataset.from_dict(
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": targets["input_ids"],
        }
    )


train_dataset = tokenize_pairs(train_pairs)
val_dataset = tokenize_pairs(val_pairs)
test_dataset = tokenize_pairs(test_pairs)
print()
print(train_dataset)

Token indices sequence length is longer than the specified maximum sequence length for this model (545 > 512). Running this sequence through the model will result in indexing errors


input   token length  50th= 268  90th= 297  99th= 338
target  token length  50th=  30  90th=  39  99th=  57

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


## 4. Save

`07` reads these directly; raw text pairs kept for the manipulation check.


In [6]:
OUT_DIR = Path("data/processed/rehearsal_elaborative")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset.save_to_disk(str(OUT_DIR / "train"))
val_dataset.save_to_disk(str(OUT_DIR / "val"))
test_dataset.save_to_disk(str(OUT_DIR / "test"))

pd.DataFrame(train_pairs).to_csv(OUT_DIR / "train_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(val_pairs).to_csv(OUT_DIR / "val_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(test_pairs).to_csv(OUT_DIR / "test_pairs_raw.csv", index=False, encoding="utf-8-sig")

print(f"Saved to: {OUT_DIR}")
for name, ds in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    print(f"  {name}: {len(ds)} rows")

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 89788.15 examples/s] 

Saved to: data/processed/rehearsal_elaborative
  train: 3000 rows
  val: 299 rows
  test: 300 rows


## Summary

- Source: `EdinburghNLP/xsum`, 3,000/300/300. `test` touched once, end of `07`.
- Docs truncated to `max_words` (350).
- No oracle step — target is abstractive by construction.
- Next: `07`. B's novel n-gram ratio should be **high**; near-zero = silently became A.
- Stage 2 spec'd in `03_실험 설계`, blocked on QG model (`09`).
